In [1]:
import os
import time
import requests
import pandas as pd
from datetime import date
from dotenv import load_dotenv

load_dotenv() 
print ("Libraries imported")

Libraries imported


In [2]:
import sqlite3 as sql
from datetime import date, timedelta

In [3]:
connection = sql.connect("../UAS_Tracker.db")

In [4]:
today_last_week = date.today() - timedelta(days=7)
today_last_week_str = today_last_week.isoformat()

**DELETE AFTER 9/21 Weekly Update**

In [5]:
last_update = date.today() - timedelta(days=4)
last_update_str = last_update.isoformat()

In [6]:
sql_string = """ SELECT api_internal_id
                FROM contracts
                WHERE last_modified_date >= ?
                             """

In [7]:
cursor = connection.cursor()

**CHANGE FOR NEXT UPDATE**

In [8]:
cursor.execute( sql_string, (last_update_str,) )

awards = cursor.fetchall()
print(awards)

[('CONT_AWD_N6600126P6083_9700_-NONE-_-NONE-',), ('CONT_AWD_70CMSD26FC0000059_7012_70B02C24A00000030_7014',), ('CONT_AWD_70CMSD26FC0000058_7012_70B02C24A00000030_7014',), ('CONT_AWD_15DDHQ26P00001005_1524_-NONE-_-NONE-',), ('CONT_AWD_140D0426F1204_1406_140D0424A0028_1406',), ('CONT_AWD_140D0426F1201_1406_140D0424A0008_1406',), ('CONT_AWD_140D0426F1187_1406_140D0426A0008_1406',), ('CONT_AWD_1331L526F13351304_1301_NNG15SD11B_8000',)]


**Endpoint documentation found here** <br>
https://github.com/fedspendingtransparency/usaspending-api/blob/master/usaspending_api/api_contracts/contracts/v2/transactions.md

In [9]:
insert_string = """INSERT OR REPLACE INTO contract_transactions 
                (transaction_id,
                type,
                action_type,
                action_date,
                description,
                modification_number,
                federal_action_obligation,
                date_added,
                api_internal_id
                )

                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                 """

In [10]:
today = date.today()

In [11]:
for award in awards:
    request = requests.post(
    url = "https://api.usaspending.gov/api/v2/transactions/",
    json = {
        "award_id": award[0],
        "page": 1,
        "sort": "action_date",
        "order": "asc",    
        "limit": 250
    }
    )
    results = request.json()["results"]
    for result in results:
        cursor.execute (insert_string,(
        result["id"],
        result["type_description"],
        result["action_type_description"],
        result["action_date"],
        result["description"],
        result["modification_number"],
        result["federal_action_obligation"],
        today.isoformat(),
        award[0],
    ))
connection.commit()